## Train on CelebA

In [ ]:
from train import Trainer
from config import config
from model import UNet
from modules import NoiseInjectionPipeline
from denoising_diffusion import NoisePredictionPipeline, DDPM

# Forward diffusion
noise_injection_pipeline = NoiseInjectionPipeline(
    config=config,
    random=True
)

# Noise prediction backbone
unet = UNet(
    in_channels=3,
    out_channels=3,
    time_embd_dim=config["time_embed"],
    base_channels=64
)

# Noise prediction pipeline
noise_prediction_pipeline = NoisePredictionPipeline(
    config=config,
    noise_prediction_model=unet
)

# Complete DDPM
ddpm = DDPM(
    noise_injection_pipieline=noise_injection_pipeline,
    noise_prediction_pipeline=noise_prediction_pipeline
)

trainer = Trainer(
    ddpm=ddpm,
    config=config,

    dataset_path="../data/celeba/img_align_celeba",

    batch_size = 32,
    image_size = 64,

    epochs = 5000,
    learning_rate=1e-4,

    validation_split = 0.05,

    checkpoint_dir="./checkpoint",
    checkpoint_every = 10,

    project_name="DiffusionEngine",
    run_name="DDPM-CelebA-128x128",

    num_workers=4,
    use_amp=True
)
trainer.resume("./checkpoint/latest.pt")
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Himanshu Singh\_netrc.
wandb: Currently logged in as: himanshusr451tehs (himanshusr451tehs-personaluse-com) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


DDPM TRAINER INITIALIZED
Device:          cuda
Mixed precision: True
Dataset size:    23429
Training images: 22258
Validation:      1171
Batch size:      32
Image size:      64x64
Training steps:  695 per epoch
W&B run:         https://wandb.ai/himanshusr451tehs-personaluse-com/DiffusionEngine/runs/wjh3gscw
TRAINING RESUMED
Starting epoch: 144
Global step:    99385
Best val loss:  0.016703


Epoch 144/5000: 100%|██████████| 695/695 [03:01<00:00,  3.83it/s, loss=0.011463, lr=1.00e-06]



Epoch:       144/5000
Train Loss:  0.019504
Val Loss:    0.019440
Grad Norm:   0.050398
LR:          1.00e-06
Checkpoint saved: checkpoint\latest.pt


Epoch 145/5000: 100%|██████████| 695/695 [02:58<00:00,  3.90it/s, loss=0.014565, lr=1.00e-06]



Epoch:       145/5000
Train Loss:  0.018970
Val Loss:    0.019568
Grad Norm:   0.049728
LR:          1.00e-06
Checkpoint saved: checkpoint\latest.pt


Epoch 146/5000:  24%|██▎       | 164/695 [00:40<02:05,  4.22it/s, loss=0.011612, lr=1.00e-06]

## Sample from the Model

In [ ]:
import torch
from sample import sample
import matplotlib.pyplot as plt

checkpoint_pth = "./checkpoint/best.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_name = torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU Execution"

# Load the best checkpoint
checkpoint = torch.load(
    checkpoint_pth,
    map_location=device,
    weights_only=False
)
saved_config = checkpoint.get("config", "No config metadata found in checkpoint")

# Restore model weights
ddpm.load_state_dict(checkpoint["model_state_dict"])
ddpm.to(device)
ddpm.eval()

print("=" * 60)
print("DDPM IMAGE GENERATION PIPELINE INITIALIZED")
print("=" * 60)
print(f"Execution Device : {device.type.upper()} ({device_name})")
print(f"Checkpoint Path  : {checkpoint_pth}")
print(f"Loaded From Step : {checkpoint.get('global_step', 'Unknown')}")
print(f"Checkpoint Loss  : {checkpoint.get('best_val_loss', 'Unknown')}")
print("-" * 60)
print("DDPM MODEL CONFIGURATION DETAILS:")
print(saved_config)
print("-" * 60)
print("Sampling and generating image... Please wait.")
print("=" * 60)

generated_image = sample(ddpm, config, device)
generated_image = (generated_image + 1.0) / 2.0 # Convert [-1, 1] -> [0, 1]
generated_image = generated_image.clamp(0, 1) # for visualization in matplotlib

# Convert BCHW -> HWC
image = generated_image[0].permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Generated Image")
plt.show()

##